# Lab 2 - Three-arm Model Router Hill Climbing

## Scenario

Northstar Devices must determine whether routing can lower inference cost and latency without reducing policy compliance or account-security quality. Compare the fixed GPT deployment, GPT-family Model Router, and open-weight Model Router with the same Aurora X1 core and challenge cases in the same order.

This notebook makes **60 billable live requests**. There is no mock path. The 20-case workload is a controlled smoke comparison, not sufficient evidence for production promotion.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import display

repo_root = Path.cwd() if (Path.cwd() / ".env").exists() else Path.cwd().parent
load_dotenv(repo_root / ".env")

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
BASELINE_GPT_DEPLOYMENT = os.getenv("BASELINE_GPT_DEPLOYMENT", "")
GPT_FAMILY_ROUTER_DEPLOYMENT = os.getenv("GPT_FAMILY_ROUTER_DEPLOYMENT", "")
OPEN_WEIGHT_ROUTER_DEPLOYMENT = os.getenv("OPEN_WEIGHT_ROUTER_DEPLOYMENT", "")

required = {
    "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
    "BASELINE_GPT_DEPLOYMENT": BASELINE_GPT_DEPLOYMENT,
    "GPT_FAMILY_ROUTER_DEPLOYMENT": GPT_FAMILY_ROUTER_DEPLOYMENT,
    "OPEN_WEIGHT_ROUTER_DEPLOYMENT": OPEN_WEIGHT_ROUTER_DEPLOYMENT,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise RuntimeError(f"Missing live configuration: {', '.join(missing)}")

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 110})
print("Execution mode: LIVE MICROSOFT FOUNDRY")
for name, deployment in required.items():
    print(f"{name}: {deployment}")

## 1. Controlled arms and workload

| Arm | Deployment role | Controlled change |
|---|---|---|
| `fixed_gpt_baseline` | Direct GPT model | No routing |
| `gpt_family_router` | GPT-family Model Router | Routing within the recorded GPT allowlist |
| `open_weight_router` | Open-weight Model Router | Routing within three recorded alternatives |

Every arm receives the same 12 core and 8 challenge cases in the same order. Compare challenge and high-risk slices as hard constraints before considering token-cost or latency improvements. Deployment manifests from Lab 0 remain the authoritative eligible-model evidence.

In [ ]:
dataset_paths = [
    repo_root / "lab-1-live-evaluation" / "router_eval.jsonl",
    repo_root / "lab-1-live-evaluation" / "router_eval_challenge.jsonl",
]
records = [
    json.loads(line)
    for path in dataset_paths
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
dataset = pd.DataFrame(records)

assert len(dataset) == 20 and dataset.request_id.is_unique
assert dataset.dataset_split.value_counts().to_dict() == {"core": 12, "challenge": 8}
assert dataset.groupby("task").size().to_dict() == {
    "classification": 5,
    "reasoning": 5,
    "retrieval": 5,
    "summarisation": 5,
}
assert dataset.risk_tier.value_counts().to_dict() == {"standard": 12, "high": 8}
display(dataset[["request_id", "dataset_split", "workflow", "risk_tier", "task"]])
print("Controlled evaluation rows: 20 (12 core + 8 challenge)")

## 2. Execute the live comparison

Each request records the deployment arm, service-reported underlying model, token usage, response text, and end-to-end wall-clock latency. Prompt and response text remain in memory only and are excluded from exported evidence.

In [ ]:
arms = {
    "fixed_gpt_baseline": BASELINE_GPT_DEPLOYMENT,
    "gpt_family_router": GPT_FAMILY_ROUTER_DEPLOYMENT,
    "open_weight_router": OPEN_WEIGHT_ROUTER_DEPLOYMENT,
}

live_rows = []
with (
    DefaultAzureCredential(process_timeout=60) as credential,
    AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    for arm, deployment in arms.items():
        for item in dataset.itertuples(index=False):
            started = time.perf_counter()
            response = openai_client.responses.create(model=deployment, input=item.prompt)
            usage = getattr(response, "usage", None)
            live_rows.append({
                "arm": arm,
                "deployment": deployment,
                "request_id": item.request_id,
                "dataset_split": item.dataset_split,
                "workflow": item.workflow,
                "risk_tier": item.risk_tier,
                "task": item.task,
                "ground_truth": item.ground_truth,
                "response": getattr(response, "output_text", "") or "",
                "selected_model": getattr(response, "model", "unknown"),
                "input_tokens": getattr(usage, "input_tokens", None),
                "output_tokens": getattr(usage, "output_tokens", None),
                "latency_ms": (time.perf_counter() - started) * 1000,
            })
            row = live_rows[-1]
            print(f"{arm} {item.request_id}: {row['selected_model']} | {row['latency_ms']:.0f} ms")

comparison_results = pd.DataFrame(live_rows)
if len(comparison_results) != 60 or comparison_results.response.eq("").any():
    raise RuntimeError("All 60 live requests must produce non-empty responses.")
if comparison_results[["input_tokens", "output_tokens"]].isna().any().any():
    raise RuntimeError("Token usage is missing from one or more responses.")
print("Completed 60 live Foundry calls.")

In [ ]:
def terms(value):
    return re.findall(r"[a-z0-9]+", str(value).lower())


def overlap_metrics(response, ground_truth):
    response_terms = terms(response)
    truth_terms = terms(ground_truth)
    response_counts = {term: response_terms.count(term) for term in set(response_terms)}
    truth_counts = {term: truth_terms.count(term) for term in set(truth_terms)}
    overlap = sum(min(response_counts.get(term, 0), truth_counts.get(term, 0)) for term in truth_counts)
    precision = overlap / max(1, len(response_terms))
    recall = overlap / max(1, len(truth_terms))
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    coverage = len(set(response_terms) & set(truth_terms)) / max(1, len(set(truth_terms)))
    return precision, recall, f1, coverage

metric_rows = comparison_results.apply(
    lambda row: overlap_metrics(row.response, row.ground_truth), axis=1, result_type="expand"
)
metric_rows.columns = ["precision", "recall", "f1", "reference_coverage"]
comparison_results = pd.concat([comparison_results, metric_rows], axis=1)
comparison_results["exact_match"] = (
    comparison_results.response.str.strip().str.lower()
    == comparison_results.ground_truth.str.strip().str.lower()
)
comparison_results["quality_pass"] = np.where(
    comparison_results.task.eq("classification"),
    comparison_results.exact_match,
    comparison_results.f1.ge(0.50),
)
display(comparison_results[[
    "arm", "request_id", "task", "selected_model", "input_tokens", "output_tokens",
    "latency_ms", "f1", "reference_coverage", "quality_pass",
]].round(3))

## 3. Compare quality, tokens, latency, and routing

These are point-in-time observations from a small controlled workload. Lexical scores are transparent regression indicators, not safety, groundedness, factuality, or human-acceptance evaluations. Latency includes client and network time and is not a service benchmark.

In [ ]:
arm_summary = comparison_results.groupby("arm").agg(
    requests=("request_id", "count"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    mean_reference_coverage=("reference_coverage", "mean"),
    total_input_tokens=("input_tokens", "sum"),
    total_output_tokens=("output_tokens", "sum"),
    latency_p50_ms=("latency_ms", "median"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
    selected_models=("selected_model", "nunique"),
).round(3)

by_slice = comparison_results.groupby(["arm", "dataset_split", "risk_tier"]).agg(
    requests=("request_id", "count"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    total_input_tokens=("input_tokens", "sum"),
    total_output_tokens=("output_tokens", "sum"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
).round(3)

by_task = comparison_results.groupby(["arm", "task"]).agg(
    requests=("request_id", "count"),
    quality_pass_rate=("quality_pass", "mean"),
    mean_f1=("f1", "mean"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    latency_p50_ms=("latency_ms", "median"),
    latency_p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
).round(3)

display(arm_summary, by_slice, by_task)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.barplot(data=comparison_results, x="task", y="f1", hue="arm", errorbar=None, ax=axes[0, 0])
axes[0, 0].axhline(0.50, color="firebrick", linestyle="--", label="Local F1 floor")
axes[0, 0].set_title("Observed mean F1 by task")
axes[0, 0].tick_params(axis="x", rotation=20)

token_totals = comparison_results.groupby("arm")[["input_tokens", "output_tokens"]].sum().reset_index()
token_long = token_totals.melt(id_vars="arm", var_name="token_type", value_name="tokens")
sns.barplot(data=token_long, x="arm", y="tokens", hue="token_type", ax=axes[0, 1])
axes[0, 1].set_title("Observed total tokens")
axes[0, 1].tick_params(axis="x", rotation=20)

latency = arm_summary[["latency_p50_ms", "latency_p95_ms"]].reset_index().melt(
    id_vars="arm", var_name="percentile", value_name="latency_ms"
)
sns.barplot(data=latency, x="arm", y="latency_ms", hue="percentile", ax=axes[1, 0])
axes[1, 0].set_title("Observed end-to-end latency")
axes[1, 0].tick_params(axis="x", rotation=20)

selection = comparison_results.groupby(["arm", "selected_model"]).size().rename("requests").reset_index()
sns.barplot(data=selection, x="arm", y="requests", hue="selected_model", ax=axes[1, 1])
axes[1, 1].set_title("Service-reported model selections")
axes[1, 1].tick_params(axis="x", rotation=20)
axes[1, 1].legend(fontsize=7)

plt.tight_layout()
plt.show()
print("Point-in-time smoke observations; not service benchmarks.")

## 4. Validate observed routes

The selected-model field is runtime evidence, while the Lab 0 deployment manifests are authoritative subset evidence. An observed model outside its configured family invalidates this comparison. A model need not appear in only 20 prompts for the deployment manifest to remain valid.

In [ ]:
gpt_allowed = {
    "gpt-5.6-sol", "gpt-5.6-terra", "gpt-5.6-luna", "gpt-5.5", "gpt-5.4",
    "gpt-5.4-mini", "gpt-5.4-nano", "gpt-5.2", "gpt-5", "gpt-5-mini", "gpt-5-nano",
}
open_weight_allowed = {
    "gpt-oss-120b", "Llama-4-Maverick-17B-128E-Instruct-FP8", "DeepSeek-V3.2",
}
expected_subsets = {
    "gpt_family_router": gpt_allowed,
    "open_weight_router": open_weight_allowed,
}


def belongs_to_model_family(reported_model, allowed_models):
    return any(
        reported_model == model or reported_model.startswith(f"{model}-")
        for model in allowed_models
    )


route_validation = {}
for arm, allowed in expected_subsets.items():
    observed = set(comparison_results.loc[comparison_results.arm.eq(arm), "selected_model"])
    unexpected = {model for model in observed if not belongs_to_model_family(model, allowed)}
    route_validation[arm] = not unexpected
    print(f"{arm} observed: {', '.join(sorted(observed))}")
    if unexpected:
        raise RuntimeError(f"{arm} returned out-of-subset models: {sorted(unexpected)}")

print("Observed router selections are within their recorded subsets.")
print("Deployment manifests remain authoritative for models not selected by this workload.")

## 5. Decision gate and redacted evidence

Test the business objective in the correct order: first reject any router that reduces queue accuracy, challenge-set quality, or high-risk policy compliance relative to the fixed baseline. Only candidates that retain those hard constraints may claim an improvement from fewer tokens, lower dated price, or lower repeated latency.

Do not declare a production winner from this smoke workload. Promotion requires at least 100 representative prompts, independent safety and model-graded evaluation, current price data, repeated latency samples, and human review of material failures.

In [ ]:
baseline = arm_summary.loc["fixed_gpt_baseline"]
baseline_rows = comparison_results.query("arm == 'fixed_gpt_baseline'")
baseline_high_risk_pass = baseline_rows.query("risk_tier == 'high'").quality_pass.mean()
baseline_challenge_pass = baseline_rows.query("dataset_split == 'challenge'").quality_pass.mean()

delta_rows = []
for arm in ["gpt_family_router", "open_weight_router"]:
    candidate = arm_summary.loc[arm]
    candidate_rows = comparison_results.query("arm == @arm")
    candidate_high_risk_pass = candidate_rows.query("risk_tier == 'high'").quality_pass.mean()
    candidate_challenge_pass = candidate_rows.query("dataset_split == 'challenge'").quality_pass.mean()
    delta_rows.append({
        "candidate": arm,
        "quality_pass_rate_delta": candidate.quality_pass_rate - baseline.quality_pass_rate,
        "challenge_pass_rate_delta": candidate_challenge_pass - baseline_challenge_pass,
        "high_risk_pass_rate_delta": candidate_high_risk_pass - baseline_high_risk_pass,
        "mean_f1_delta": candidate.mean_f1 - baseline.mean_f1,
        "total_tokens_delta": (
            candidate.total_input_tokens + candidate.total_output_tokens
            - baseline.total_input_tokens - baseline.total_output_tokens
        ),
        "latency_p95_ms_delta": candidate.latency_p95_ms - baseline.latency_p95_ms,
        "quality_constraints_retained": bool(
            candidate.quality_pass_rate >= baseline.quality_pass_rate
            and candidate_challenge_pass >= baseline_challenge_pass
            and candidate_high_risk_pass >= baseline_high_risk_pass
        ),
    })
deltas = pd.DataFrame(delta_rows).set_index("candidate").round(3)
display(deltas)

smoke_gates = pd.Series({
    "all_60_requests_completed": len(comparison_results) == 60,
    "all_responses_non_empty": comparison_results.response.ne("").all(),
    "all_token_usage_present": comparison_results[["input_tokens", "output_tokens"]].notna().all().all(),
    "each_arm_has_20_requests": comparison_results.groupby("arm").size().eq(20).all(),
    "all_four_tasks_in_each_arm": comparison_results.groupby("arm").task.nunique().eq(4).all(),
    "core_and_challenge_in_each_arm": comparison_results.groupby("arm").dataset_split.nunique().eq(2).all(),
    "observed_routes_within_subsets": all(route_validation.values()),
    "sample_large_enough_for_promotion": dataset.shape[0] >= 100,
    "safety_evaluation_completed": False,
})
display(smoke_gates.to_frame("passed"))
print("OBSERVE ONLY: no arm is promoted from this 20-case smoke comparison.")

artifact_dir = repo_root / "lab-2-hill-climbing" / "model_router_artifacts"
artifact_dir.mkdir(exist_ok=True)
redacted_columns = [
    "arm", "deployment", "request_id", "dataset_split", "workflow", "risk_tier", "task",
    "selected_model", "input_tokens", "output_tokens", "latency_ms", "precision", "recall",
    "f1", "reference_coverage", "quality_pass",
]
comparison_results[redacted_columns].to_csv(
    artifact_dir / "redacted_three_arm_results.csv", index=False,
)
arm_summary.to_csv(artifact_dir / "three_arm_summary.csv")
manifest = {
    "scenario": "northstar-aurora-x1-launch-support",
    "execution_mode": "live",
    "requests": int(len(comparison_results)),
    "dataset_rows_per_arm": int(len(dataset)),
    "dataset_splits": dataset.dataset_split.value_counts().sort_index().to_dict(),
    "risk_tiers": dataset.risk_tier.value_counts().sort_index().to_dict(),
    "arms": arms,
    "observed_models": {
        arm: sorted(group.selected_model.unique())
        for arm, group in comparison_results.groupby("arm")
    },
    "sample_size_warning": "Controlled smoke comparison only; not sufficient for promotion.",
}
(artifact_dir / "three_arm_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8",
)
print("Wrote redacted artifacts:")
for path in sorted(artifact_dir.iterdir()):
    print(" -", path.name)

## What you learned

1. The business objective is conditional: lower cost and latency count only when policy compliance and account-security quality do not regress.
2. Core, challenge, high-risk, and task slices expose trade-offs hidden by an overall average.
3. A fixed-model baseline distinguishes routing gains from general model capability.
4. A 20-case live comparison validates the experiment, but cannot support production promotion.

Previous: [Lab 1 - Live evaluation](../lab-1-live-evaluation/README.md) | [Workshop home](../README.md)